In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/toilatoan2409/test-dataset/README.dataset.txt
/kaggle/input/datasets/toilatoan2409/test-dataset/README.roboflow.txt
/kaggle/input/datasets/toilatoan2409/test-dataset/valid/Blast_PCRM_R12-0316a_png.rf.e776ad6dfb40474fed6255f0f26c1703.jpg
/kaggle/input/datasets/toilatoan2409/test-dataset/valid/Blast_PCRM_R13-0081B_png.rf.8857b8c0921f3b07a3cba5e3b445b8ee.jpg
/kaggle/input/datasets/toilatoan2409/test-dataset/valid/Blast_PCRM_R14-0379_png.rf.6bb045f802c4e9f2521ded494f62076d.jpg
/kaggle/input/datasets/toilatoan2409/test-dataset/valid/Blast_PCRM_d5-29_png.rf.6c1f55c53ccd51124ddcbb2769d2231c.jpg
/kaggle/input/datasets/toilatoan2409/test-dataset/valid/Blast_PCRM_R13-0036B_png.rf.a4b1a239b8d6b438829506e216a0ba2c.jpg
/kaggle/input/datasets/toilatoan2409/test-dataset/valid/Blast_PCRM_R14-0068_png.rf.30499bb4930e5615b59abf458ac86c2e.jpg
/kaggle/input/datasets/toilatoan2409/test-dataset/valid/Blast_PCRM_R14-0044_png.rf.c34b75b649e167bf78ea348c106db8c8.jpg
/kaggle/input/dataset

In [9]:
!cp -r /kaggle/input/datasets/toilatoan2409/src-test1/src_final /kaggle/working/
%cd /kaggle/working

/kaggle/working


In [10]:
!find /kaggle -name "tasdf" -type d

/kaggle/input/datasets/toilatoan2409/src-test1/src_final/src/tasdf
/kaggle/working/src_final/src/tasdf


In [13]:
import sys
from pathlib import Path

# Thay đường dẫn bên dưới bằng thư mục CHỨA thư mục con 'tasdf' của bạn
REPO_DIR = "/kaggle/working/src_final/src"   # Ví dụ: nếu tasdf nằm tại /kaggle/working/tasdf_src/tasdf

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# Bây giờ import sẽ thành công tuyệt đối:
from tasdf import TASDF

print("Đã nạp thành công thư viện TASDF:", TASDF)

Đã nạp thành công thư viện TASDF: <class 'tasdf.api.TASDF'>


In [15]:
import os
from pathlib import Path
from tasdf import TASDF

# 1. Đường dẫn dữ liệu từ thư mục /kaggle/input
# (Lưu ý: /kaggle/input là READ-ONLY, chỉ để đọc ảnh và file nhãn)
DATASET_DIR = "/kaggle/input/datasets/toilatoan2409/test-dataset"  # Thay bằng tên dataset thật của bạn

data_cfg = {
    "format": "coco",                      # hoặc "folder" nếu là thư mục ảnh/mask
    "path": DATASET_DIR,
    "train": "train",                      # tên thư mục con train
    "val": "valid",                        # tên thư mục con valid (hoặc "val")
    "image_size": 256,
    "labeled_ratio": 0.2,                  # Tỷ lệ 20% ảnh có nhãn, 80% không nhãn
}

# 2. Khởi tạo mô hình TASDF
# Bạn có thể truyền cấu hình tùy chỉnh hoặc dùng mặc định
model = TASDF()

# 3. Kích hoạt huấn luyện
# Toàn bộ output (train.log, metrics.json, checkpoint best.pt/latest.pt) 
# sẽ tự động ghi vào /kaggle/working/runs (phân vùng có quyền GHI)
results = model.train(
    data=data_cfg,
    epochs=50,
    batch=8,                               # Kaggle GPU T4/P100 có 16GB VRAM, tăng batch lên 8 hoặc 16
    imgsz=256,
    lr=1e-4,
    device="cuda",
    amp=True,                              # Kích hoạt FP16 tối ưu hóa GPU Tensor Cores
    num_workers=2,                         # Khuyến nghị 2 trên Kaggle để tránh tràn /dev/shm
    project="/kaggle/working/runs/train",
    name="exp_kaggle",
)

print("Huấn luyện hoàn tất! File checkpoint tốt nhất:", results["best_checkpoint"])

[06:27:11] Bắt đầu huấn luyện: Epoch 1 -> 50 | Thiết bị: cuda | AMP: True

---------------------------------------------------------------------------------------------------------------------
Epoch    | GPU_Mem   | L_total   | L_sup     | L_dist    | Val_Dice   | ZP_Dice   | TE_Dice   | ICM_Dice  | BC_Dice  
---------------------------------------------------------------------------------------------------------------------
01/50    | 6.82G     | 1.0894    | 2.1787    | 0.0077    | 43.72%     | 63.3%     | 80.0%     | 0.0%      | 31.6%    
02/50    | 6.82G     | 0.7006    | 1.4011    | 0.0063    | 60.00%     | 74.1%     | 83.1%     | 36.8%     | 46.1%    
03/50    | 6.82G     | 0.5451    | 1.0902    | 0.0071    | 66.63%     | 71.6%     | 79.5%     | 15.5%     | 100.0%   
04/50    | 6.82G     | 0.4513    | 0.9025    | 0.0054    | 82.51%     | 76.3%     | 85.9%     | 67.8%     | 100.0%   
05/50    | 6.82G     | 0.3828    | 0.7656    | 0.0044    | 83.53%     | 77.0%     | 84.4%     | 72.

In [20]:
# 1. Đánh giá chi tiết mDice / mIoU từng lớp trên tập validation
val_metrics = model.val(data=data_cfg, branch="net1")

# 2. Dự đoán thử nghiệm trên ảnh bất kỳ và bóc tách TE / Blastocoel
predictions = model.predict(
    source=f"{DATASET_DIR}/test",          # Hoặc đường dẫn 1 file ảnh cụ thể
    imgsz=256,
    output_dir="/kaggle/working/runs/predict",
    decompose=False,                        # Bật thuật toán phân rã hình thái học
    te_thickness_ratio=0.06,
    save=True,
)


  KẾT QUẢ ĐÁNH GIÁ ĐỘ LẬP | TẬP: [VAL] | NHÁNH: [net1]
Cấu trúc Phôi học            | Dice (%)     | IoU / Jaccard (%) 
-----------------------------------------------------------------
Zona Pellucida (ZP)          | 82.71        | 72.43             
Trophectoderm (TE)           | 88.95        | 83.38             
Inner Cell Mass (ICM)        | 80.83        | 70.22             
Blastocyst Cavity (BC)       | 100.00       | 100.00            
-----------------------------------------------------------------
TRUNG BÌNH (mDice / mIoU)    | 88.12        | 81.51             

   [+] 11_4AA_jpg.rf.1e72be14287d587007fc8d9469cf4857: {'Zona Pellucida (ZP)': 7301, 'Trophectoderm (TE)': 26134, 'Inner Cell Mass (ICM)': 5251}
   [+] 12_3AA_jpg.rf.9e8ab6033841dc101d16e7da6a034b93: {'Zona Pellucida (ZP)': 9060, 'Trophectoderm (TE)': 16127, 'Inner Cell Mass (ICM)': 2817}
   [+] 14_3AA_jpg.rf.ff07fc96e11ee7abcf589e142833f058: {'Zona Pellucida (ZP)': 6847, 'Trophectoderm (TE)': 30509}
   [+] 16_4AA_jpg

In [19]:
!zip -r /kaggle/working/tasdf_kaggle_run.zip /kaggle/working/runs

  adding: kaggle/working/runs/ (stored 0%)
  adding: kaggle/working/runs/train/ (stored 0%)
  adding: kaggle/working/runs/train/exp_kaggle/ (stored 0%)
  adding: kaggle/working/runs/train/exp_kaggle/train.log (deflated 63%)
  adding: kaggle/working/runs/train/exp_kaggle2/ (stored 0%)
  adding: kaggle/working/runs/train/exp_kaggle2/train.log (deflated 78%)
  adding: kaggle/working/runs/train/exp_kaggle2/checkpoints/ (stored 0%)
  adding: kaggle/working/runs/train/exp_kaggle2/checkpoints/checkpoint_best.pt (deflated 8%)
  adding: kaggle/working/runs/train/exp_kaggle2/checkpoints/latest.pt (deflated 8%)
  adding: kaggle/working/runs/train/exp_kaggle2/checkpoints/last.pt (deflated 8%)
  adding: kaggle/working/runs/train/exp_kaggle2/checkpoints/best.pt (deflated 8%)
  adding: kaggle/working/runs/train/exp_kaggle2/checkpoints/checkpoint_latest.pt (deflated 8%)
  adding: kaggle/working/runs/train/exp_kaggle2/metrics.json (deflated 81%)
  adding: kaggle/working/runs/predict/ (stored 0%)
  addi

In [17]:
!python src_final/src/eval.py --config /kaggle/working/src_final/src/configs/coco_blastocyst.yaml --checkpoint /kaggle/working/checkpoints/tasdf_coco_blastocyst_best.pt --split test


usage: eval.py [-h] [--config CONFIG] --weights WEIGHTS
               [--split {val,valid,test,train}]
               [--branch {net1,net2,ensemble}] [--imgsz IMGSZ]
               [--device DEVICE] [--save_json SAVE_JSON]
eval.py: error: the following arguments are required: --weights


In [6]:
!python src/src/inference.py --config /kaggle/working/src/src/configs/coco_blastocyst.yaml --checkpoint /kaggle/working/checkpoints/tasdf_coco_blastocyst_best.pt --image_dir /kaggle/input/datasets/toilatoan2409/test-dataset/test --output_dir /kaggle/working/test_proposals

>> Nạp thành công checkpoint từ: /kaggle/working/checkpoints/tasdf_coco_blastocyst_best.pt
>> Đang suy luận trên 76 ảnh, kết quả lưu tại: /kaggle/working/test_proposals/
   [+] Blast_PCRM_1201754-D5_png.rf.7b9ad5c4a6691fb59474a414902234dd: Phát hiện các vùng: Zona Pellucida (ZP), Trophectoderm (TE), Inner Cell Mass (ICM)
   [+] Blast_PCRM_R13-0009A_png.rf.704abc1bbba186edc9e4915ef0c44984: Phát hiện các vùng: Zona Pellucida (ZP), Trophectoderm (TE), Inner Cell Mass (ICM)
   [+] 12_3AA_jpg.rf.9e8ab6033841dc101d16e7da6a034b93: Phát hiện các vùng: Zona Pellucida (ZP), Trophectoderm (TE), Inner Cell Mass (ICM)
   [+] 5_3BA_jpg.rf.5128ff406970abe75316783bed89c624: Phát hiện các vùng: Zona Pellucida (ZP), Trophectoderm (TE), Inner Cell Mass (ICM)
   [+] Blast_PCRM_R13-0140a_png.rf.deb161195e29e9c528ba1eeb512f87df: Phát hiện các vùng: Zona Pellucida (ZP), Trophectoderm (TE), Inner Cell Mass (ICM)
   [+] Blast_PCRM_R13-0009B_png.rf.418d1602bb283d360aeac744795268a3: Phát hiện các vùng: Zona Pell

In [9]:
!zip -r /kaggle/working/test_proposals.zip /kaggle/working


updating: kaggle/working/test_proposals/ (stored 0%)
updating: kaggle/working/test_proposals/Blast_PCRM_1202833-D5_png.rf.e05cc6191c196cd564f7eca74d7897d4_overlay.png (deflated 8%)
updating: kaggle/working/test_proposals/Blast_PCRM_R13-0009B_png.rf.418d1602bb283d360aeac744795268a3_mask_raw.png (deflated 18%)
updating: kaggle/working/test_proposals/2_3BB_jpg.rf.7017150b4761f07411758f8e476a868b_mask_raw.png (deflated 45%)
updating: kaggle/working/test_proposals/14_3AA_jpg.rf.ff07fc96e11ee7abcf589e142833f058_composite.png (deflated 16%)
updating: kaggle/working/test_proposals/2_3AA_jpg.rf.4112f41d1763f062df23f8d9a955e0c6_composite.png (deflated 16%)
updating: kaggle/working/test_proposals/Blast_PCRM_R13-0235b_png.rf.14bf0e7b2a631348ccaf2c44384bc162_overlay.png (deflated 9%)
updating: kaggle/working/test_proposals/25_3AA_jpg.rf.c1aba4f1779b91d31436ec80b1cb8255_composite.png (deflated 17%)
updating: kaggle/working/test_proposals/12_3AA_jpg.rf.9e8ab6033841dc101d16e7da6a034b93_composite.png (